# µIR Cookbook

This notebook is the practical <s>notebook</s> cookbook for canonical µIR.

It combines two perspectives:

- frontend: parsing C into µIR and inspecting the result
- middleend: running passes and prototyping external µIR transforms


## Setup

Run this cell first. It locates the repository root, adds `src/` to `sys.path`, and imports the main µIR helpers used throughout the nookbook.


In [ ]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
import sys
import textwrap

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src" / "uhls").exists():
    repo_root = repo_root.parent

if not (repo_root / "src" / "uhls").exists():
    raise RuntimeError("could not locate repo root containing src/uhls")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from uhls.frontend import lower_source_to_uir
from uhls.interpreter import run_uir
from uhls.middleend.passes.opt import ConstPropPass, CopyPropPass, DCEPass, SimplifyCFGPass, UnrollLoopsPass, CanonicalizeLoopsPass
from uhls.middleend.passes.util import PassContext, PassManager
from uhls.middleend.uir import BinaryOp, Block, ConstOp, Function, Module, ReturnOp, UnaryOp
from uhls.middleend.uir import format_module, pretty, verify_function, verify_module

print(f"repo_root = {repo_root}")


## Frontend: From C To µIR

The frontend chapter starts with a tiny C example, lowers it into canonical µIR, and then inspects the resulting blocks and instructions.


In [ ]:
c_source = textwrap.dedent(
    """
    int32_t add_then_scale(int32_t x) {
        int32_t y;
        y = x + 1;
        return y * 2;
    }
    """
).strip()

module = lower_source_to_uir(c_source)
print(format_module(module))


In [ ]:
verify_module(module)
function = module.functions[0]
result = run_uir(function, {"x_0": 7})
print("return_value =", result.return_value)


### Integrated `main` Testbenches And `uhls_printf`

For small examples, µhLS can keep the testbench next to the kernel in the same µC source file. The synthesizable function stays as a normal callable kernel, while `main(void)` provides concrete inputs, checks the result, and returns a status code.

When lowered to µIR, `main` becomes ordinary control flow with function-local arrays, direct `call` operations, and `print` operations for `uhls_printf(...)`. The CLI follows the same convention: if a module has multiple functions and one is named `main`, `./uhls run file.uir` selects `main` automatically. Hardware-backed runs use the same idea after the kernel has been lowered and wrapped, so calls from `main` become generated invocations of the synthesized design.


In [ ]:
testbench_source = textwrap.dedent(
    """
    int32_t dot4_relu(int32_t A[4], int32_t B[4]) {
        int32_t i;
        int32_t sum = 0;
        for (i = 0; i < 4; i = i + 1) {
            sum = sum + A[i] * B[i];
        }
        if (sum < 0) {
            sum = 0;
        }
        return sum;
    }

    int32_t main(void) {
        int32_t A[4] = {1, 2, 3, 4};
        int32_t B[4] = {4, 3, 2, 1};
        int32_t expected = 20;
        int32_t res;
        res = dot4_relu(A, B);
        if (res != expected) {
            uhls_printf("Unexpected return value, is [%d] expected [%d]", res, expected);
            return 1;
        }
        uhls_printf("Success!");
        return 0;
    }
    """
).strip()

testbench_module = lower_source_to_uir(testbench_source)
verify_module(testbench_module)
print(format_module(testbench_module))

main_function = testbench_module.get_function("main")
result = run_uir(main_function, module=testbench_module)
print("return_value =", result.return_value)
print("stdout =", result.state.stdout)


### Things To Inspect

As you read the printed µIR, look for:

- block labels and explicit control flow
- SSA value names
- one final terminator per block
- how loads, stores, calls, and returns are represented


## Middleend: Built-In Passes

Before writing a custom pass, it helps to see how built-in µIR pass pipelines behave.


In [ ]:
pipeline = [ConstPropPass(), CopyPropPass(), DCEPass(), SimplifyCFGPass()]
optimized = PassManager(pipeline).run(module, PassContext())
verify_module(optimized)
print(format_module(optimized))


## Middleend: A Minimal External Pass

An external pass can be a callable or a class with `run(...)`. The safest pattern is to start from a `deepcopy` and rebuild only the pieces you want to change.


In [ ]:
class RenameModulePass:
    name = "rename_module"

    def run(self, ir):
        result = deepcopy(ir)
        result.name = "notebook_demo"
        return result

renamed = RenameModulePass().run(module)
print(format_module(renamed))


## Middleend: Algebraic Identity Prototype

This section mirrors the style of `src/uhls/external_passes/algebraic_identity.py`, but keeps the logic inline so you can experiment before exporting it.


In [ ]:
class AlgebraicIdentityNotebookPass:
    name = "algebraic_identity_notebook"

    def run(self, ir):
        result = deepcopy(ir)
        result.functions = [self._rewrite_function(function) for function in result.functions]
        return result

    def _rewrite_function(self, function):
        return Function(
            name=function.name,
            params=deepcopy(function.params),
            blocks=[
                Block(
                    label=block.label,
                    instructions=[self._rewrite_instruction(instr) for instr in block.instructions],
                    terminator=deepcopy(block.terminator),
                )
                for block in function.blocks
            ],
            return_type=function.return_type,
            entry=function.entry,
            local_arrays=deepcopy(function.local_arrays),
        )

    def _rewrite_instruction(self, instruction):
        if isinstance(instruction, BinaryOp) and instruction.opcode == "add" and instruction.rhs == 0:
            return UnaryOp("mov", instruction.dest, instruction.type, instruction.lhs)
        if isinstance(instruction, BinaryOp) and instruction.opcode == "mul" and instruction.rhs == 1:
            return UnaryOp("mov", instruction.dest, instruction.type, instruction.lhs)
        if isinstance(instruction, BinaryOp) and instruction.opcode == "mul" and instruction.rhs == 0:
            return ConstOp(instruction.dest, instruction.type, 0)
        return deepcopy(instruction)

identity_module = Module(
    functions=[
        Function(
            name="demo",
            params=[],
            return_type="i32",
            blocks=[
                Block(
                    "entry",
                    instructions=[
                        ConstOp("x", "i32", 7),
                        BinaryOp("add", "a", "i32", "x", 0),
                        BinaryOp("mul", "b", "i32", "a", 1),
                        BinaryOp("mul", "c", "i32", "b", 0),
                    ],
                    terminator=ReturnOp("c"),
                )
            ],
        )
    ]
)

rewritten = AlgebraicIdentityNotebookPass().run(identity_module)
verify_module(rewritten)
print(format_module(rewritten))


## Middleend: CFG And `phi` Pitfalls

If you change CFG structure, you must keep `phi` incoming predecessor labels in sync with the real CFG predecessors of the destination block.

A good rule of thumb is:

- instruction-only rewrites are easy
- block cloning and branch rewrites are where you need to slow down
- always validate with `verify_module(...)` and `./uhls lint ...`


### Inspecting CFG And DFG and CDFG Structure

The helpers below are useful when you want to understand the shape of a function before writing a transform. They give you a lightweight way to inspect control-flow and dataflow without leaving the notebook.


In [ ]:
from uhls.middleend.passes.analyze import build_cfg, build_dfg
from uhls.middleend.passes.util.dot import to_cfg_dot, to_dfg_dot
from graphviz import Source

cfg = build_cfg(function)
print("CFG order:", cfg.order)
print("CFG successors:")
for label in cfg.order:
    print(f"  {label} -> {sorted(cfg.successors[label])}")

cfg_dot = to_cfg_dot(function)
print("\nCFG DOT preview:\n")
print("\n".join(cfg_dot.splitlines()[:20]))
Source(cfg_dot) # use graphiz to render in notebook

In [ ]:
from uhls.middleend.passes.analyze import build_dfg
from uhls.middleend.passes.util.dot import to_dfg_dot

dfg = build_dfg(function)
first_block = function.blocks[0].label
block_dfg = dfg.blocks[first_block]

print(f"Inspecting DFG for block: {first_block}")
print("DFG nodes:")
for node in block_dfg.nodes:
    print(f"  {node.id}: opcode={node.opcode}, defines={node.defines}, uses={node.uses}")

print("\nDFG edges:")
for edge in block_dfg.edges:
    print(f"  {edge.source} -> {edge.target} [{edge.kind}:{edge.label}]")

dfg_dot = to_dfg_dot(dfg)
print("\nDFG DOT preview:\n")
print("\n".join(dfg_dot.splitlines()[:24]))
Source(dfg_dot) # use graphiz to render in notebook


In [ ]:
from uhls.middleend.passes.util.dot import to_cdfg_dot

cdfg_dot = to_cdfg_dot(function)
print("\nCDFG DOT preview:\n")
print("\n".join(cdfg_dot.splitlines()[:24]))
Source(cdfg_dot) # use graphiz to render in notebook


### More CFG Rewrite Recipes For µIR

Below are two practical CFG rewrite patterns:

- structural cleanup with `simplify_cfg_function(...)`
- manual predecessor-label rewrites that also update `phi` nodes

The main lesson is that CFG rewrites are usually easy to describe, but only safe when the block graph and the `phi` incoming labels stay consistent.


In [ ]:
from uhls.middleend.passes.opt.simplify_cfg import simplify_cfg_function
from uhls.middleend.uir import BranchOp

cfg_cleanup_function = Function(
    name="cleanup_demo",
    params=[],
    return_type="i32",
    blocks=[
        Block("entry", terminator=BranchOp("jump")),
        Block("jump", terminator=BranchOp("work")),
        Block("work", instructions=[ConstOp("x", "i32", 7)], terminator=BranchOp("exit")),
        Block("exit", terminator=ReturnOp("x")),
        Block("dead", instructions=[ConstOp("y", "i32", 99)], terminator=ReturnOp("y")),
    ],
)

print("Before simplify_cfg:\n")
print(pretty(cfg_cleanup_function))
Source(to_cfg_dot(cfg_cleanup_function))


In [ ]:
cfg_cleanup_simplified = simplify_cfg_function(cfg_cleanup_function)
verify_function(cfg_cleanup_simplified)

print("After simplify_cfg:\n")
print(pretty(cfg_cleanup_simplified))
print("return_value =", run_uir(cfg_cleanup_simplified, {}).return_value)
Source(to_cfg_dot(cfg_cleanup_simplified))


#### Recipe: Renaming A Predecessor And Updating `phi`

This is one of the most common CFG maintenance tasks. The broken version below renames a predecessor block `left_jump` and updates branch targets, but forgets to rewrite the `phi` incoming label. Verification should fail.


In [ ]:
from dataclasses import replace
from uhls.middleend.uir import CondBranchOp, Parameter, PhiOp

# the original "correct" version with left_jump labels
phi_demo = Function(
    name="phi_demo",
    params=[Parameter("sel", "i1")],
    return_type="i32",
    blocks=[
        Block("entry", terminator=CondBranchOp("sel", "left", "right")),
        Block("left", instructions=[ConstOp("x_left", "i32", 1)], terminator=BranchOp("left_jump")),
        Block("left_jump", terminator=BranchOp("merge")),
        Block("right", instructions=[ConstOp("x_right", "i32", 2)], terminator=BranchOp("merge")),
        Block(
            "merge",
            instructions=[PhiOp("x", "i32", [("left_jump", "x_left"), ("right", "x_right")])],
            terminator=ReturnOp("x"),
        ),
    ],
)

print(pretty(phi_demo))
Source(to_cfg_dot(phi_demo))


In [ ]:
broken_phi_demo = deepcopy(phi_demo)

# pseudo rewrite of block label left_jump -> left_tail and targets, but missing update of phi instructions consuming the old left_jump label
for block in broken_phi_demo.blocks:
    if block.label == "left_jump":
        block.label = "left_tail"
    if isinstance(block.terminator, BranchOp) and block.terminator.target == "left_jump":
        block.terminator.target = "left_tail"
    elif isinstance(block.terminator, CondBranchOp):
        if block.terminator.true_target == "left_jump":
            block.terminator.true_target = "left_tail"
        if block.terminator.false_target == "left_jump":
            block.terminator.false_target = "left_tail"

try:
    verify_function(broken_phi_demo)
except Exception as exc:
    print(type(exc).__name__)
    print(exc)
    print("Comment: Exception is expected, as phi node still expects label `left_jump`")

Source(to_cfg_dot(broken_phi_demo))


In [ ]:
def rewrite_phi_predecessor_labels(function, old_label, new_label):
    result = deepcopy(function)
    for block in result.blocks:
        for instruction in block.instructions:
            if getattr(instruction, "opcode", None) != "phi":
                break
            instruction.incoming = [
                replace(item, pred=new_label) if item.pred == old_label else item
                for item in instruction.incoming
            ]
    return result

# the fixed version with left_tail labels rewritten in instructions
fixed_phi_demo = rewrite_phi_predecessor_labels(broken_phi_demo, old_label="left_jump", new_label="left_tail")
verify_function(fixed_phi_demo)
print(pretty(fixed_phi_demo))
Source(to_cfg_dot(fixed_phi_demo))
print("sel=1 ->", run_uir(fixed_phi_demo, {"sel": 1}).return_value)
print("sel=0 ->", run_uir(fixed_phi_demo, {"sel": 0}).return_value)

Source(to_cfg_dot(fixed_phi_demo))

## Middleend: Larger Loop Transformation Example

Loop transforms are where µIR rewrites become noticeably more subtle. Even a conservative loop unroll needs to reason about:

- the loop header block
- loop-carried `phi` nodes
- the latch / body block
- fresh SSA names for cloned instructions
- new predecessor labels for the header `phi` nodes
- preserving the exit behavior when the trip count is not a multiple of the unroll factor

In this section we define a conservative `unroll_by_2` transform directly inside the notebook. The goal is not to handle every possible loop shape, but to show how a real loop rewrite can be assembled from canonical µIR pieces.


In [ ]:
loop_source = textwrap.dedent(
    """
    int32_t sum_to_8(void) {
        int32_t i;
        int32_t acc;
        acc = 0;
        for (i = 0; i < 8; i = i + 1) {
            acc = acc + i;
        }
        return acc;
    }
    """
).strip()

loop_module = lower_source_to_uir(loop_source)
verify_module(loop_module)
loop_function = loop_module.functions[0]

print(format_module(loop_module))
print("return_value =", run_uir(loop_function, {}).return_value)


In [ ]:
from uhls.middleend.passes.analyze import detect_loops

loop_cfg = build_cfg(loop_function)
loops = detect_loops(loop_function)

print("Loop CFG order:", loop_cfg.order)
for label in loop_cfg.order:
    print(f"  {label} -> {sorted(loop_cfg.successors[label])}")

print("\nDetected loops:")
for loop in loops:
    print(f"  header={loop.header}, latches={loop.latches}, body={sorted(loop.body)}, exits={sorted(loop.exits)}")

Source(to_cfg_dot(loop_function))


### What To Notice In The Lowered Loop

The frontend usually lowers a simple `for` loop into this conservative canonical shape:

- an entry block that initializes induction and accumulator values
- a header block containing `phi` nodes and the loop condition
- one loop body / latch block that branches back to the header
- one exit block that consumes the final loop-carried value

That regularity is exactly why a narrow notebook-defined loop transform can work. We will only target one very specific shape:

- one header with exactly two predecessors and two successors
- one unique latch/body block that branches back to the header
- loop-carried values represented by header `phi` nodes
- straight-line body code that we can clone once with fresh SSA names


In [ ]:
from uhls.middleend.passes.analyze import control_flow
from uhls.middleend.uir import CallOp, CompareOp, IncomingValue, LoadOp, ParamOp, PhiOp, PrintOp, StoreOp, Variable

pipeline = [UnrollLoopsPass("for_header_1", 2)]
unrolled_module = PassManager(pipeline).run(loop_module, PassContext())
unrolled_function = unrolled_module.get_function("sum_to_8")

verify_module(unrolled_module)
print(format_module(unrolled_module))
print("return_value =", run_uir(unrolled_function, {}).return_value)

Source(to_cfg_dot(unrolled_function))


In [ ]:

pipeline = [CanonicalizeLoopsPass(), ConstPropPass(), CopyPropPass(), DCEPass()]
canon_module = PassManager(pipeline).run(unrolled_module, PassContext())
verify_module(canon_module)
print(format_module(canon_module))
Source(to_cfg_dot(canon_module.get_function("sum_to_8")))


### Loop Transform Takeaways

A conservative loop transform in canonical µIR usually needs these ingredients:

- identify the loop header and the unique latch/body block
- read the loop-carried values from the header `phi` nodes
- clone instructions with fresh SSA result names
- remap cloned operands through a name mapping
- add new predecessor labels to the header `phi` nodes
- re-check the transformed module with `verify_module(...)`

The important notebook lesson is that this is all ordinary Python over the µIR data model. Once the transform works interactively, you can lift the same code into a standalone external pass module.
